# SIRCmw Phase Portrait


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import matplotlib
font = {'size' : 15}
matplotlib.rc('font', **font)

In [5]:
#  fixed parameters 
mu    = 0.02       
alpha = 365/3  
delta = 1/1.61  
gamma = 0.35 
sigma = 0.07874   
beta0 = 600.0  

# eps = tilde_eps / (S0 * I0)
S0_ref, I0_ref = 0.2, 0.001
SI_ref = S0_ref * I0_ref  


def sircmw_rhs(t, y, eps1, eps2):
    S, I, R, C = np.maximum(y, 0.0)
    dS = mu*(1 - S) - beta0*S*I + (1 + eps2*S*I)*gamma*C
    dI = beta0*S*I + sigma*beta0*C*I - (mu + alpha)*I
    dR = (1 - sigma)*beta0*C*I + alpha*I - mu*R - (1 + eps1*S*I)*delta*R
    dC = (1 + eps1*S*I)*delta*R - beta0*C*I - mu*C - (1 + eps2*S*I)*gamma*C
    return np.array([dS, dI, dR, dC])

In [ ]:
# (R,I)

fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
plt.close()


def phase_portrait(tilde_eps=0.0):
    ax.cla()
    eps = tilde_eps / SI_ref

    # Integrate the system to get trajectory and equilibrium
    y0 = [0.2, 0.001, 0.499, 0.3]
    sol = solve_ivp(sircmw_rhs, (0.0, 50.0), y0, args=(eps, eps), method='RK45', rtol=1e-4, atol=1e-6)
    S_traj, I_traj, R_traj, C_traj = sol.y
    S_star, I_star, R_star, C_star = S_traj[-1], I_traj[-1], R_traj[-1], C_traj[-1]

    # Grid setup
    R = np.linspace(0, 1, 50)
    I = np.linspace(0, 1, 50)
    R_2d, I_2d = np.meshgrid(R, I)
    rem = np.maximum(1.0 - I_2d - R_2d, 0) # remainder to split 
    
    # Proportional equilibrium scaling for remaining variables S and C
    sum_SC = S_star + C_star
    if sum_SC > 1e-12:
        S_grid = rem * (S_star / sum_SC)
        C_grid = rem * (C_star / sum_SC)
    else:
        S_grid = rem / 2.0
        C_grid = rem / 2.0

    rhs = sircmw_rhs(0.0, [S_grid, I_2d, R_2d, C_grid], eps, eps)
    dR_2d, dI_2d = rhs[2], rhs[1]
    dR_2d[rem <= 0] = np.nan
    dI_2d[rem <= 0] = np.nan
    # we use streamplot to draw the flow direction field
    ax.streamplot(R, I, dR_2d, dI_2d, broken_streamlines=False, density=0.8)

    # Plot integrated trajectory and equilibrium
    ax.plot(R_traj, I_traj, color='red', ls='--', lw=2.5, label='Trajectory')
    ax.scatter([R_traj[0]], [I_traj[0]], color='green', s=60, edgecolor='k', zorder=6, label='Start')
    ax.scatter([R_star], [I_star], color='red', marker='*', s=120, edgecolor='k', zorder=7, label='Equilibrium')

    ax.set_xlabel('R (Recovered)')
    ax.set_ylabel('I (Infected)')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('image')
    ax.legend(loc='upper right')

    fig.canvas.draw_idle()
    plt.close()
    return fig


interact(phase_portrait,
         tilde_eps=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.0,
                               description='ε̃',
                               continuous_update=False))


interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='ε̃', max=2.0, step=0.05), O…

<function __main__.phase_portrait(tilde_eps=0.0)>

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 7), dpi=300)
plt.close()


def phase_portrait_SI(tilde_eps=0.0):
    ax2.cla()
    eps = tilde_eps / SI_ref

    # Integrate the system to get trajectory and equilibrium
    y0 = [0.2, 0.001, 0.499, 0.3]
    sol = solve_ivp(sircmw_rhs, (0.0, 50.0), y0, args=(eps, eps), method='RK45', rtol=1e-4, atol=1e-6)
    S_traj, I_traj, R_traj, C_traj = sol.y
    S_star, I_star, R_star, C_star = S_traj[-1], I_traj[-1], R_traj[-1], C_traj[-1]

    # Grid setup
    S = np.linspace(0, 1, 50)
    I = np.linspace(0, 1, 50)
    S_2d, I_2d = np.meshgrid(S, I)
    rem = np.maximum(1.0 - I_2d - S_2d, 0)
    
    # Proportional equilibrium scaling for remaining variables R and C
    sum_RC = R_star + C_star
    if sum_RC > 1e-12:
        R_grid = rem * (R_star / sum_RC)
        C_grid = rem * (C_star / sum_RC)
    else:
        R_grid = rem / 2.0
        C_grid = rem / 2.0

    rhs = sircmw_rhs(0.0, [S_2d, I_2d, R_grid, C_grid], eps, eps)
    dS_2d, dI_2d = rhs[0], rhs[1]
    dS_2d[rem <= 0] = np.nan
    dI_2d[rem <= 0] = np.nan
    ax2.streamplot(S, I, dS_2d, dI_2d, broken_streamlines=False, density=0.8)

    # Plot integrated trajectory and equilibrium
    ax2.plot(S_traj, I_traj, color='red', ls='--', lw=2.5, label='Trajectory')
    ax2.scatter([S_traj[0]], [I_traj[0]], color='green', s=60, edgecolor='k', zorder=6, label='Start')
    ax2.scatter([S_star], [I_star], color='red', marker='*', s=120, edgecolor='k', zorder=7, label='Equilibrium')

    ax2.set_xlabel('S (Susceptible)')
    ax2.set_ylabel('I (Infected)')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('image')
    ax2.legend(loc='upper right')

    fig2.canvas.draw_idle()
    plt.close()
    return fig2


interact(phase_portrait_SI,
         tilde_eps=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.0,
                               description='ε̃',
                               continuous_update=False))


interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='ε̃', max=2.0, step=0.05), O…

<function __main__.phase_portrait_SI(tilde_eps=0.0)>